## College Football Notebook

### Setup

Data is from https://collegefootballdata.com/exporter/stats/player/season

In [22]:
import pandas as pd
import os
from pathlib import PurePath
import pickle
import subprocess

TOP_LEVEL_DIR = PurePath(os.getcwd())

RAW_DATA_DIR = PurePath(TOP_LEVEL_DIR / "cfb-data")
CLEANED_DATA_DIR = PurePath(TOP_LEVEL_DIR / "cleaned-data")

### Combining all the data together and cleaning it

In [27]:
schema = {
    "Season": int,
    "PlayerId": int,
    "Player": str,
    "Team": str,
    "Conference": str,
    "Category": str,
    "StatType": str,
    "Stat": float,
}

bad_player_names = ("Team", "Kohnercullimore", "- ")


def is_valid_player(player: str) -> bool:
    if player == " ":
        return False
    else:
        is_bad_player_name = not any(
            [
                bad_player_name
                for bad_player_name in bad_player_names
                if bad_player_name in player
            ]
        )
        return is_bad_player_name


# combine the csv's
subprocess.run(
    ["cat cfb-data/*.csv > cleaned-data/cfb_data.csv"], shell=True, check=True
)

all_data = pd.read_csv(PurePath(CLEANED_DATA_DIR / "cfb_data.csv"), low_memory=False)
all_data = all_data[
    all_data.apply(
        lambda row: is_valid_player(row["Player"])
        and row["Season"] != '\ufeff"Season"',
        axis=1,
    )
]
all_data = all_data.astype(schema)
all_data.to_csv(PurePath(CLEANED_DATA_DIR / "cfb_data.csv"), index=False)

### Aggregating player stats for multiple years and keeping track of it

In [28]:
data_to_transfer = data_to_transfer = {
    "RAW_DATA_DIR": RAW_DATA_DIR,
    "CLEANED_DATA_DIR": CLEANED_DATA_DIR,
    "TOP_LEVEL_DIR": TOP_LEVEL_DIR,
}
pickle.dump(
    data_to_transfer, open(PurePath(TOP_LEVEL_DIR / "cfb_data_to_transfer.pkl"), "wb")
)

In [38]:
!python cfb-data-combine.py

print("Data combined")

Data combined
